In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import HfFileSystem, hf_hub_download

In [ ]:
# 1. Connect to the Hugging Face filesystem
fs = HfFileSystem()
repo_id = "pdearena/NavierStokes-2D-conditoned" # Note the typo in their repo name

print(f"Listing files in {repo_id}...")

# 2. Find all .h5 simulation files in the repository
# The files are stored under the 'datasets/' prefix in the HfFileSystem
files = fs.ls(f"datasets/{repo_id}", detail=False)
h5_files = [f for f in files if f.endswith('.h5')]

if not h5_files:
    print("No .h5 files found!")
    # return
    
# We will just grab the very first trajectory to test
first_file_name = h5_files[0].split("/")[-1]

print(f"\nDownloading 1 trajectory: {first_file_name}")
print("This might take a minute depending on your connection...")

# 3. Download the file and cache it locally
file_path = hf_hub_download(
    repo_id=repo_id, 
    repo_type="dataset", 
    filename=first_file_name
)

print(f"\nDownload complete! File saved to cache at: {file_path}")


In [ ]:
def find_first_dataset(name, obj):
    """Callback to find the first actual dataset inside the HDF5 groups."""
    if isinstance(obj, h5py.Dataset):
        return obj
    return None

def print_h5_structure(name, obj):
    """Prints the name and shape of every dataset in the file."""
    if isinstance(obj, h5py.Dataset):
        print(f" - {name}: {obj.shape}")

In [ ]:
# 4. Open the HDF5 file and inspect the physics
with h5py.File(file_path, 'r') as f:
    print("\n--- Internal HDF5 File Structure ---")
    def print_h5_structure(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f" - {name}: {obj.shape}")
    f.visititems(print_h5_structure)
    print("------------------------------------\n")
    
    # Scope-safe trick: use a list to store the found path
    found_paths = []
    
    def find_physics_tensor(name, obj):
        if isinstance(obj, h5py.Dataset) and len(obj.shape) >= 3:
            found_paths.append(name)
            return True # Stops the visititems loop

    f.visititems(find_physics_tensor)
    
    if not found_paths:
        print("Could not find a spatio-temporal tensor (>= 3 dimensions) in this file.")
    else:
        data_path = found_paths[0]
        print(f"Selected PDE tensor: '{data_path}'")
        
        # Load the trajectory
        trajectory = f[data_path][()]
        print(f"Trajectory shape: {trajectory.shape}")
        
        time_len = trajectory.shape[0]
        time_steps_to_plot = [0, time_len // 4, time_len // 2, 3 * time_len // 4, time_len - 1]
        
        fig, axes = plt.subplots(1, len(time_steps_to_plot), figsize=(20, 4))
        fig.suptitle(f"PDEArena Data - Tensor: {data_path}", fontsize=16)
        
        for ax, t in zip(axes, time_steps_to_plot):
            # Dynamically extract a 2D field based on the tensor's dimensions
            if len(trajectory.shape) == 3: 
                field = trajectory[t, :, :]
            elif len(trajectory.shape) == 4: 
                # Grab the last channel (usually buoyancy in this dataset)
                field = trajectory[t, -1, :, :]
            elif len(trajectory.shape) == 5: 
                # Shape: (Time, Condition, Channels, X, Y)
                field = trajectory[t, 0, -1, :, :]
            else:
                field = trajectory[t]
            
            im = ax.imshow(field, cmap='inferno', origin='lower')
            ax.set_title(f"Step {t}")
            ax.axis('off')
            
        cbar = fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.015, pad=0.04)
        cbar.set_label("Field Value")
        
        plt.show()